In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
class SafetyMLP(nn.Module):
    def __init__(self, input_size):
        super(SafetyMLP, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.layers(x)

In [ ]:
features_df = pd.read_csv('../../../data/features_for_model.csv').dropna()
X = features_df.drop(columns=['label']).values
y = features_df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

train_loader = DataLoader(
    TensorDataset(
        torch.FloatTensor(X_train_scaled),
        torch.LongTensor(y_train)
    ),
    batch_size=32,
    shuffle=True
)

print(f"Załadowano {len(features_df)} wierszy.")
print(f"Kształt wejścia: {X_train_scaled.shape}")

In [ ]:
test_tensor = torch.FloatTensor(X_test_scaled)
test_y_tensor = torch.FloatTensor(y_test).view(-1, 1)

In [ ]:
model = SafetyMLP(X_train_scaled.shape[1])
criterion = nn.BCELoss()
smoothing = 0.1
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)

best_loss = float('inf')
alpha = 0.1

for epoch in range(50):
    model.train()
    train_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()

        smoothed_y = batch_y * (1 - alpha) + (alpha / 2)

        outputs = model(batch_x)
        loss = criterion(outputs, smoothed_y)

        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    with torch.no_grad():
        test_preds = model(test_tensor)
        test_loss = criterion(test_preds, test_y_tensor)

        scheduler.step(test_loss)

        if test_loss < best_loss:
            best_loss = test_loss
            torch.save(model.state_dict(), '../local_models/safety_mlp_new_bert.pth')
            print(f"Epoch {epoch+1}: Nowy najlepszy model! Test Loss: {test_loss:.4f}")

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/50, Train Loss: {train_loss/len(train_loader):.4f}")

In [ ]:
os.makedirs('../local_models', exist_ok=True)
torch.save(model.state_dict(), '../local_models/safety_mlp_new_bert.pth')